[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/MNPS_Likelihood_Evaluator.ipynb)

# **MNPS Job Classification Likelihood Evaluator**

## Overview
Evaluates how closely model classifications match human evaluator expectations using a comprehensive **0-5 Likelihood Score** based on:
- KSAC similarity between roles
- Salary impact analysis
- Time required to correct errors

### DSI + MNPS Collaboration

### Required Inputs:
- `Sample JDs.csv` - Original job descriptions
- `Job_Classifications_Batch.csv` - Model classification results
- `Evaluation Resources.zip` - Contains:
  - Ground Truth Masterfile.csv
  - MNPS KSACs.csv
  - MNPS_Role_Groups_by_KSAC_Similarity_FINAL.csv
  - salary_by_major_role_grouping.csv
  - Time to correct an error in hours.csv

### Output:
- Likelihood Score per record (0-5 scale)
- Batch average score
- Detailed severity breakdown
- All results saved to Google Drive with timestamp

### Scoring Guide:
- **5.0**: Exact match
- **4.0-4.9**: Same similarity group, low salary difference
- **3.0-3.9**: Same similarity group, high salary difference
- **2.0-2.9**: Different groups, quick correction time
- **0.0-1.9**: Different groups, long correction time

In [ ]:
#============================================
# ENVIRONMENT SETUP
#============================================

import os
import json
import zipfile
import datetime as dt
from pathlib import Path
import pandas as pd
import numpy as np
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Create timestamped output folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
output_base = Path("/content/drive/My Drive/Likelihood Assessment/Run Results")
run_folder = output_base / f"RUN_{timestamp}"
run_folder.mkdir(parents=True, exist_ok=True)

print(f"✅ Environment setup complete")
print(f"📁 Output folder: {run_folder}")
print(f"🕐 Timestamp: {timestamp}")

In [ ]:
#============================================
# ROBUST CSV LOADING FUNCTION
#============================================

def load_csv_with_fallback(filepath, description="file"):
    """Load CSV with multiple encoding attempts and error handling"""
    
    print(f"\n📄 Loading {description}...")
    
    # Convert to Path object if string
    filepath = Path(filepath)
    
    if not filepath.exists():
        raise FileNotFoundError(f"❌ File not found: {filepath}")
    
    # List of encodings to try
    encodings = ['utf-8', 'latin1', 'iso-8859-1', 'cp1252', 'utf-16', 'windows-1252']
    
    # List of delimiters to try
    delimiters = [',', ';', '\t', '|']
    
    # Try each encoding
    for encoding in encodings:
        # Try each delimiter
        for delimiter in delimiters:
            try:
                if delimiter == ',':
                    print(f"   Trying {encoding} encoding...")
                else:
                    print(f"   Trying {encoding} encoding with '{delimiter}' delimiter...")
                
                df = pd.read_csv(filepath, encoding=encoding, sep=delimiter)
                
                # Verify we got actual data (at least 2 columns)
                if len(df.columns) >= 2:
                    if delimiter == ',':
                        print(f"   ✅ Successfully loaded with {encoding} encoding")
                    else:
                        print(f"   ✅ Successfully loaded with {encoding} encoding and '{delimiter}' delimiter")
                    print(f"   📊 Loaded {len(df)} rows with {len(df.columns)} columns")
                    return df
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue
            except Exception:
                continue
    
    # Last resort: try with error handling
    try:
        print(f"   Attempting with error handling...")
        df = pd.read_csv(filepath, encoding='utf-8', errors='ignore', on_bad_lines='skip')
        print(f"   ⚠️ Loaded with error handling (some characters may be lost)")
        print(f"   📊 Loaded {len(df)} rows with {len(df.columns)} columns")
        return df
    except Exception as e:
        raise Exception(f"❌ Failed to load {description} after trying all encodings: {e}")

print("✅ Robust CSV loading function initialized")
print("   Supports: UTF-8, Latin1, ISO-8859-1, CP1252, UTF-16, Windows-1252")
print("   Delimiters: comma, semicolon, tab, pipe")

In [ ]:
#============================================
# FILE DISCOVERY AND LOADING
#============================================

# Expected input files
SAMPLE_JDS = Path("/content/Sample JDs.csv")
MODEL_RESULTS = Path("/content/Job_Classifications_Batch.csv")
EVAL_ZIP = Path("/content/Evaluation Resources.zip")

# Check for files in Drive as fallback
drive_base = Path("/content/drive/My Drive")

print("🔍 Discovering input files...")

# Look for Sample JDs
if not SAMPLE_JDS.exists():
    drive_sample = drive_base / "Sample JDs.csv"
    if drive_sample.exists():
        SAMPLE_JDS = drive_sample
        print(f"✅ Found Sample JDs in Drive: {SAMPLE_JDS}")
    else:
        raise FileNotFoundError(f"❌ Missing: Sample JDs.csv (check /content/ or Drive)")
else:
    print(f"✅ Found Sample JDs: {SAMPLE_JDS}")

# Look for Model Results
if not MODEL_RESULTS.exists():
    drive_results = drive_base / "Job_Classifications_Batch.csv"
    if drive_results.exists():
        MODEL_RESULTS = drive_results
        print(f"✅ Found Classifications in Drive: {MODEL_RESULTS}")
    else:
        raise FileNotFoundError(f"❌ Missing: Job_Classifications_Batch.csv (check /content/ or Drive)")
else:
    print(f"✅ Found Classifications: {MODEL_RESULTS}")

# Look for Evaluation Resources
if not EVAL_ZIP.exists():
    drive_zip = drive_base / "Evaluation Resources.zip"
    if drive_zip.exists():
        EVAL_ZIP = drive_zip
        print(f"✅ Found Evaluation Resources in Drive: {EVAL_ZIP}")
    else:
        raise FileNotFoundError(f"❌ Missing: Evaluation Resources.zip (check /content/ or Drive)")
else:
    print(f"✅ Found Evaluation Resources: {EVAL_ZIP}")

# Extract Evaluation Resources
print("\n📦 Extracting Evaluation Resources...")
eval_dir = Path("/content/eval_resources")
eval_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(EVAL_ZIP, 'r') as z:
    z.extractall(eval_dir)
    print(f"✅ Extracted to: {eval_dir}")

# Load all resources with robust CSV handling
print("\n📂 Loading all data files...")
jd_df = load_csv_with_fallback(SAMPLE_JDS, 'Sample JDs')
model_df = load_csv_with_fallback(MODEL_RESULTS, 'Job Classifications')

# Find and load evaluation resource files
eval_files = {}
for file in eval_dir.rglob('*.csv'):
    file_lower = file.name.lower()
    if 'ground' in file_lower or 'truth' in file_lower:
        eval_files['ground_truth'] = file
    elif 'ksac' in file_lower and 'role' not in file_lower:
        eval_files['ksacs'] = file
    elif 'role' in file_lower and 'group' in file_lower and 'similarity' in file_lower:
        eval_files['similarity'] = file
    elif 'salary' in file_lower:
        eval_files['salary'] = file
    elif 'time' in file_lower:
        eval_files['time'] = file

# Load evaluation resources
gt_df = load_csv_with_fallback(eval_files['ground_truth'], 'Ground Truth') if 'ground_truth' in eval_files else pd.DataFrame()
ksacs_df = load_csv_with_fallback(eval_files['ksacs'], 'MNPS KSACs') if 'ksacs' in eval_files else pd.DataFrame()
sim_df = load_csv_with_fallback(eval_files['similarity'], 'Role Similarity Groups') if 'similarity' in eval_files else pd.DataFrame()
salary_df = load_csv_with_fallback(eval_files['salary'], 'Salary Data') if 'salary' in eval_files else pd.DataFrame()
time_df = load_csv_with_fallback(eval_files['time'], 'Time Correction Data') if 'time' in eval_files else pd.DataFrame()

print("\n✅ All data files loaded successfully!")
print(f"   Job descriptions: {len(jd_df)}")
print(f"   Model results: {len(model_df)}")
print(f"   Ground truth: {len(gt_df)}")
print(f"   KSACs: {len(ksacs_df)}")
print(f"   Similarity groups: {len(sim_df)}")
print(f"   Salary data: {len(salary_df)}")
print(f"   Time correction: {len(time_df)}")

In [ ]:
#============================================
# BUILD LOOKUP MAPS
#============================================

print("🔨 Building lookup maps...\n")

# 1. Ground truth lookup: source_row_index → expected_major_role_group
gt_lookup = {}
for _, row in gt_df.iterrows():
    idx = row.get('source_row_index')
    if pd.notna(idx):
        try:
            gt_lookup[int(idx)] = str(row.get('expected_major_role_group', '')).strip()
        except (ValueError, TypeError):
            continue

print(f"✅ Ground truth lookup: {len(gt_lookup)} entries")

# 2. KSAC similarity groups
similarity_groups = {}
for _, row in sim_df.iterrows():
    role = str(row.get('major_role_group', '')).strip()
    group_id = row.get('similarity_group_id')
    if role and pd.notna(group_id):
        try:
            similarity_groups[role] = int(group_id)
        except (ValueError, TypeError):
            continue

print(f"✅ Similarity groups: {len(similarity_groups)} role mappings")

# 3. Salary lookup: role → avg_salary
salary_lookup = {}
for _, row in salary_df.iterrows():
    role = str(row.get('major_role_group', '')).strip()
    avg = row.get('avg_salary')
    if role and pd.notna(avg):
        try:
            salary_lookup[role] = float(avg)
        except (ValueError, TypeError):
            continue

print(f"✅ Salary lookup: {len(salary_lookup)} role mappings")

# 4. Time to correct lookup: (true_role, pred_role) → hours
time_lookup = {}
for _, row in time_df.iterrows():
    true_r = str(row.get('true_role', '')).strip()
    pred_r = str(row.get('predicted_role', '')).strip()
    t = row.get('time_hours')
    if true_r and pred_r and pd.notna(t):
        try:
            time_lookup[(true_r, pred_r)] = float(t)
        except (ValueError, TypeError):
            continue

print(f"✅ Time correction lookup: {len(time_lookup)} role pair mappings")

print("\n✅ All lookup maps built successfully!")

In [ ]:
#============================================
# LIKELIHOOD SCORE COMPUTATION
#============================================

def get_time_to_correct(true_role, pred_role):
    """Get estimated time to correct classification error"""
    # Try exact lookup
    if (true_role, pred_role) in time_lookup:
        return time_lookup[(true_role, pred_role)]
    
    # Fallback: average time for this true_role
    times = [t for (tr, pr), t in time_lookup.items() if tr == true_role]
    if times:
        return np.mean(times)
    
    # Final fallback
    return 1.0  # default 1 hour

def compute_record_likelihood(model_role, expected_role):
    """Compute likelihood score (0-5) for a single classification"""
    
    model_role = str(model_role).strip()
    expected_role = str(expected_role).strip()

    # Exact match = perfect score
    if model_role == expected_role:
        return 5.0

    # Check similarity group
    model_group = similarity_groups.get(model_role)
    expected_group = similarity_groups.get(expected_role)

    if model_group is not None and expected_group is not None and model_group == expected_group:
        # Same similarity group → use salary delta to assess severity
        avg_model = salary_lookup.get(model_role, 0)
        avg_expected = salary_lookup.get(expected_role, 1)  # avoid div/0
        
        if avg_expected == 0:
            avg_expected = 1
        
        if avg_model == 0:
            # No salary data → assume moderate error
            return 4.0
        
        salary_diff_pct = abs(avg_model - avg_expected) / avg_expected
        
        if salary_diff_pct <= 0.10:
            return 4.5  # Very similar roles
        elif salary_diff_pct <= 0.25:
            return 4.0  # Similar roles, moderate difference
        else:
            return 3.5  # Same group but significant salary difference
    else:
        # Different groups → use time to correct as severity indicator
        t = get_time_to_correct(expected_role, model_role)
        
        if t <= 0.5:
            return 2.5  # Quick fix
        elif t <= 2.0:
            return 1.5  # Moderate time to correct
        else:
            return 0.5  # Long time to correct

print("✅ Likelihood computation functions initialized")
print("\n📊 Scoring Guide:")
print("   5.0: Exact match")
print("   4.0-4.9: Same similarity group, low-moderate salary difference")
print("   3.0-3.9: Same similarity group, high salary difference")
print("   2.0-2.9: Different groups, quick correction (≤30 min)")
print("   1.0-1.9: Different groups, moderate correction (≤2 hours)")
print("   0.0-0.9: Different groups, long correction (>2 hours)")

In [ ]:
#============================================
# EVALUATE ALL RECORDS
#============================================

print("\n🎯 Evaluating classifications...\n")

results = []
skipped = 0

for idx, row in model_df.iterrows():
    source_idx = row.get('source_row_index')
    
    if pd.isna(source_idx):
        skipped += 1
        continue
    
    try:
        source_idx = int(source_idx)
    except (ValueError, TypeError):
        skipped += 1
        continue
    
    if source_idx not in gt_lookup:
        skipped += 1
        continue  # no ground truth

    model_role = row.get('major_role_group', '')
    expected_role = gt_lookup[source_idx]

    if pd.isna(model_role) or not expected_role:
        skipped += 1
        continue

    likelihood = compute_record_likelihood(model_role, expected_role)

    results.append({
        'source_row_index': source_idx,
        'job_title_original': row.get('job_title_original', ''),
        'model_major_role': str(model_role).strip(),
        'expected_major_role': expected_role,
        'likelihood_score': likelihood,
        'is_exact_match': str(model_role).strip() == expected_role
    })
    
    # Progress indicator
    if len(results) % 10 == 0:
        print(f"  Processed {len(results)} records...")

eval_df = pd.DataFrame(results)

print(f"\n✅ Evaluated {len(eval_df)} records with ground truth")
if skipped > 0:
    print(f"⚠️ Skipped {skipped} records (no ground truth or missing data)")

In [ ]:
#============================================
# SAVE RESULTS AND GENERATE SUMMARY
#============================================

if len(eval_df) == 0:
    raise ValueError("❌ No records could be evaluated — check ground truth alignment.")

print("\n💾 Saving results...")

# Save detailed report
report_path = run_folder / "Likelihood_Score_Report.csv"
try:
    eval_df.to_csv(report_path, index=False, encoding='utf-8')
except Exception:
    eval_df.to_csv(report_path, index=False, encoding='latin1')

print(f"📁 Detailed report saved: {report_path}")

# Compute batch summary
avg_score = eval_df['likelihood_score'].mean()
exact_match_rate = eval_df['is_exact_match'].mean()

# Severity breakdown
severity_breakdown = {
    "5.0 (exact)": int((eval_df['likelihood_score'] == 5.0).sum()),
    "4.0-4.9 (similar, low cost)": int(((eval_df['likelihood_score'] >= 4.0) & (eval_df['likelihood_score'] < 5.0)).sum()),
    "3.0-3.9 (similar, high cost)": int(((eval_df['likelihood_score'] >= 3.0) & (eval_df['likelihood_score'] < 4.0)).sum()),
    "2.0-2.9 (dissimilar, fast fix)": int(((eval_df['likelihood_score'] >= 2.0) & (eval_df['likelihood_score'] < 3.0)).sum()),
    "1.0-1.9 (dissimilar, moderate fix)": int(((eval_df['likelihood_score'] >= 1.0) & (eval_df['likelihood_score'] < 2.0)).sum()),
    "0.0-0.9 (dissimilar, slow fix)": int((eval_df['likelihood_score'] < 1.0).sum())
}

summary = {
    "batch_likelihood_score": round(avg_score, 2),
    "exact_match_rate": round(exact_match_rate, 4),
    "total_evaluated_records": len(eval_df),
    "timestamp": timestamp,
    "severity_breakdown": severity_breakdown,
    "score_statistics": {
        "mean": round(eval_df['likelihood_score'].mean(), 2),
        "median": round(eval_df['likelihood_score'].median(), 2),
        "std": round(eval_df['likelihood_score'].std(), 2),
        "min": round(eval_df['likelihood_score'].min(), 2),
        "max": round(eval_df['likelihood_score'].max(), 2)
    }
}

# Save summary
summary_path = run_folder / "Batch_Summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"📁 Summary saved: {summary_path}")

# Display results
print("\n" + "="*80)
print("✅ EVALUATION COMPLETE")
print("="*80)
print(f"\n📁 Results saved to: {run_folder}")
print(f"\n📊 Batch Likelihood Score: {summary['batch_likelihood_score']}/5.0")
print(f"🎯 Exact Match Rate: {summary['exact_match_rate']*100:.1f}%")
print(f"📈 Total Evaluated: {summary['total_evaluated_records']} records")

print("\n📊 Score Statistics:")
print(f"   Mean: {summary['score_statistics']['mean']}")
print(f"   Median: {summary['score_statistics']['median']}")
print(f"   Std Dev: {summary['score_statistics']['std']}")
print(f"   Range: {summary['score_statistics']['min']} - {summary['score_statistics']['max']}")

print("\n📊 Severity Breakdown:")
for category, count in severity_breakdown.items():
    percentage = (count / len(eval_df) * 100) if len(eval_df) > 0 else 0
    print(f"   {category}: {count} records ({percentage:.1f}%)")

print("\n" + "="*80)

In [ ]:
#============================================
# VISUALIZATION (OPTIONAL)
#============================================

import matplotlib.pyplot as plt
import seaborn as sns

print("\n📈 Generating visualizations...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MNPS Likelihood Score Evaluation Results', fontsize=16, fontweight='bold')

# 1. Likelihood Score Distribution
axes[0].hist(eval_df['likelihood_score'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(avg_score, color='red', linestyle='--', 
                label=f'Mean: {avg_score:.2f}', linewidth=2)
axes[0].set_xlabel('Likelihood Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Likelihood Score Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Severity Breakdown
categories = list(severity_breakdown.keys())
counts = list(severity_breakdown.values())
colors = ['green', 'lightgreen', 'yellow', 'orange', 'red', 'darkred']

axes[1].barh(categories, counts, color=colors[:len(categories)], edgecolor='black')
axes[1].set_xlabel('Count')
axes[1].set_title('Severity Breakdown')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()

# Save visualization
viz_path = run_folder / "Likelihood_Score_Visualization.png"
plt.savefig(viz_path, dpi=300, bbox_inches='tight')
print(f"📁 Visualization saved: {viz_path}")

plt.show()

print("\n✅ All visualizations complete!")